# 15 · Spark + Apache Sedona setup and smoke test (Part D, step D0)

**What this sets up:** Apache Spark (PySpark **3.5.9**, local mode on all CPU cores) with **Apache Sedona 1.9.1**,
which adds spatial SQL (points, distances, spatial joins, KNN, GeoParquet) to Spark. Versions are pinned in
`requirements-bigdata.txt`; Sedona's jars (`sedona-spark-shaded-3.5_2.12:1.9.1`, `geotools-wrapper:1.9.1-33.5`)
are downloaded from Maven Central on first start. Spark 3.5 runs on the laptop's Java 17.

**Windows only:** Spark needs Hadoop's `winutils.exe` and `hadoop.dll` to write files. They are fetched once into
`tools/hadoop/bin/` from the widely used `cdarlint/winutils` GitHub repository (Hadoop 3.3.6 builds, matching
Spark 3.5's Hadoop 3.3 client) and set for this notebook's process only — nothing is changed system-wide.

**Colab:** the install cell adds Java 17 and the same packages; Spark's temporary files go to `/content/tmp`.

**Laptop note:** Spark's own scratch folder is put in the Windows temp folder, because the project path contains spaces and Spark's block manager then fails to start on Windows; the driver binds to 127.0.0.1. Messages at shutdown like *Failed to delete ... .jar* are a known harmless Windows quirk (the jar is still locked by Java).

The shared session code is `scripts/bigdata/spark_session.py` (`get_spark()`), used by every Part D notebook.

**Smoke tests:** (1) Sedona's great-circle distance equals the project's `haversine_m()`; (2) `ST_DWithin` agrees;
(3) the distributed nearest-neighbour join `ST_KNN` gives the same answer as brute force on 2,000 points;
(4) Parquet and GeoParquet files can be written and read.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona

In [4]:
import shutil

import pyspark
import sedona as sedona_package
from bigdata.spark_session import SEDONA_JARS, get_spark

TMP = (Path("/content/tmp") if IN_COLAB else PROJECT / "data" / "tmp") / "spark_smoke"  # test files (path with spaces on the laptop)
spark = get_spark(PROJECT)
print("PySpark", pyspark.__version__, "| Sedona", sedona_package.version, "| jars:", SEDONA_JARS)
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)

PySpark 3.5.9 | Sedona 1.9.1 | jars: org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.9.1,org.datasyslab:geotools-wrapper:1.9.1-33.5
cores: 12 | Spark UI: http://127.0.0.1:4040


## Test 1–2: distance and `ST_DWithin`

In [5]:
# Test 1: great-circle distance in Sedona = the project's haversine_m() (scripts/build_india_health_facilities.py)
from build_india_health_facilities import haversine_m

towns = [("Thiruvananthapuram", 8.5241, 76.9366), ("Kochi", 9.9312, 76.2673), ("Kozhikode", 11.2588, 75.7804)]
spark.createDataFrame(towns, "name string, lat double, lon double") \
     .selectExpr("name", "lat", "lon", "ST_Point(lon, lat) AS geom").createOrReplaceTempView("towns")
d = spark.sql("""SELECT a.name AS a, b.name AS b, a.lat AS alat, a.lon AS alon, b.lat AS blat, b.lon AS blon,
                        ST_DistanceSphere(a.geom, b.geom) AS sedona_m,
                        ST_DWithin(a.geom, b.geom, 200000, true) AS within_200km
                 FROM towns a JOIN towns b ON a.name < b.name""").toPandas()
d["haversine_m"] = haversine_m(d["alat"], d["alon"], d["blat"], d["blon"])
print(d[["a", "b", "sedona_m", "haversine_m", "within_200km"]].round(0).to_string(index=False))
assert ((d["sedona_m"] - d["haversine_m"]).abs() / d["haversine_m"] < 1e-3).all(), "distance mismatch"
# Test 2: ST_DWithin (spheroid, metres) agrees with the distance
assert (d["within_200km"] == (d["sedona_m"] <= 200000)).all(), "ST_DWithin mismatch"
print("distance and ST_DWithin: OK")

        a                  b  sedona_m  haversine_m  within_200km
    Kochi Thiruvananthapuram  172848.0     172848.0          True
    Kochi          Kozhikode  156922.0     156922.0          True
Kozhikode Thiruvananthapuram  329401.0     329401.0         False
distance and ST_DWithin: OK


## Test 3: `ST_KNN` nearest-neighbour join

In [6]:
# Test 3: Sedona's distributed nearest-neighbour join (ST_KNN) = brute force on 2,000 points x 50 facilities
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
cells = pd.DataFrame({"cid": range(2000), "lat": rng.uniform(8.3, 12.7, 2000), "lon": rng.uniform(74.9, 77.3, 2000)})
fac = pd.DataFrame({"fid": range(50), "lat": rng.uniform(8.3, 12.7, 50), "lon": rng.uniform(74.9, 77.3, 50)})
spark.createDataFrame(cells).selectExpr("cid", "ST_Point(lon, lat) AS geom").createOrReplaceTempView("cells")
spark.createDataFrame(fac).selectExpr("fid", "ST_Point(lon, lat) AS geom").createOrReplaceTempView("fac")
knn = spark.sql("SELECT c.cid, f.fid FROM cells c JOIN fac f ON ST_KNN(c.geom, f.geom, 1, true)").toPandas()
dist = haversine_m(cells["lat"].to_numpy()[:, None], cells["lon"].to_numpy()[:, None],
                   fac["lat"].to_numpy()[None, :], fac["lon"].to_numpy()[None, :])
brute = pd.Series(dist.argmin(axis=1), index=cells["cid"])
agree = (knn.set_index("cid")["fid"].sort_index() == brute.sort_index()).mean()
print(f"KNN rows: {len(knn)} | same nearest facility as brute force: {agree:.2%}")
assert len(knn) == len(cells) and agree == 1.0, "KNN mismatch"

KNN rows: 2000 | same nearest facility as brute force: 100.00%


## Test 4: Parquet and GeoParquet

In [7]:
# Test 4: write and read Parquet and GeoParquet (needs winutils/hadoop.dll on Windows)
out = TMP / "smoke"
t = spark.table("towns")
t.write.mode("overwrite").parquet(str(out / "plain"))
t.write.mode("overwrite").format("geoparquet").save(str(out / "geo"))
back = spark.read.format("geoparquet").load(str(out / "geo"))
print("GeoParquet schema:", back.schema.simpleString())
assert back.count() == 3 and spark.read.parquet(str(out / "plain")).count() == 3
print("Parquet + GeoParquet: OK")

GeoParquet schema: struct<name:string,lat:double,lon:double,geom:udt,geom_bbox:struct<xmin:double,ymin:double,xmax:double,ymax:double>>


Parquet + GeoParquet: OK


## Stop Spark and clean up

In [8]:
spark.stop()
shutil.rmtree(TMP / "smoke", ignore_errors=True)
print("all smoke tests passed; Spark stopped, test files removed")

all smoke tests passed; Spark stopped, test files removed
